In [ ]:
# main.py
from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer
from src.trajectory_predictor import TrajectoryPredictor

import numpy as np
import os
import random
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

# 현재 main.py 파일의 디렉토리를 기준으로 프로젝트 루트 경로 설정
BASE_DIR = os.getcwd()
WINDOW_SIZE = 200

SEED = 41 
os.environ["PYTHONHASHSEED"] = str(SEED)  # 파이썬 해시 결정화
os.environ["TF_DETERMINISTIC_OPS"] = "1"  # TF 결정적 커널 사용
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"  # cuDNN 결정화
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # CPU oneDNN 최적화 비활성(비결정성 방지)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

config = {
    "looking_left01.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "looking_left02.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left03.csv": {"skiprows": 250, "flag": False, "zone": 52},
    "looking_left04.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left05.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "looking_right01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right02.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_right03.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right04.csv": {"skiprows": 350, "flag": True, "zone": 52},
    
    "looking_lr01.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_lr02.csv": {"skiprows": 500, "flag": True, "zone": 52},
    
    "swing_left01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "swing_left02.csv": {"skiprows": 400, "flag": False, "zone": 52},
    "swing_left03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_left04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left07.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left08.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left12.csv": {"skiprows": 300, "flag": True, "zone": 52},

    "swing_right01.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "swing_right02.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right07.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right08.csv": {"skiprows": 500, "flag": True, "zone": 52},
    "swing_right09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right12.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "calling_left01.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left02.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left03.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left04.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left05.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left06.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    
    "calling_right01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right04.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "out_looking_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "out_looking_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "out_swing_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "out_swing_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jw_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jw_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jw_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jh_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_jh_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_jh_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "looking_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "looking_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "swing_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    "calling_hr_l01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_l02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_l03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_hr_r03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    
    
    
    
}

default_config = {"skiprows": 500, "flag": False, "zone": 52}


def main():

    # ============================================================
    # 1. 학습 데이터 로딩 및 전처리
    learn_data_paths = [
        #tester1
        #보고걷기 좌회전
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left01.csv"), # 2분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left02.csv"), # 3분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left05.csv"), # 10분 
        
        #보고걷기 우회전 
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right01.csv"), # 3분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right02.csv"), # 4분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right04.csv"), # 10분 
        
        #스윙 좌회전
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left01.csv"), # 2.5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left04.csv"), # 10분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left05.csv"), # 5분
        
        #스윙 우회전 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right01.csv"), # 2.5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right03.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right04.csv"), # 10분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right05.csv"), # 5분

        
        #전화받기 좌회전 30분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left01.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left02.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left03.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left05.csv"), # 5분 
    
        #전화받기 우회전 30분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right01.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right02.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right03.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right04.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right05.csv"), # 5분 

        
        # ============================================================
        # ============================================================
        # tester2
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l03.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r03.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l03.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r03.csv"), # 5분
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l03.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r03.csv"), # 5분
        
        # ============================================================
        # ============================================================
        # tester3 
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l03.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r03.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l03.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r03.csv"), # 5분
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l03.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r03.csv"), # 5분
        
        # ============================================================
        # ============================================================
        # tester4
        
        # 보고걷기 좌회전 15분 
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l03.csv"), # 5분
        
        # 보고걷기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r03.csv"), # 5분
        
        # 스윙 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l03.csv"), # 5분
        
        # 스윙 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r03.csv"), # 5분 
        
        # 전화받기 좌회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l03.csv"), # 5분
        
        # 전화받기 우회전 15분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r02.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r03.csv"), # 5분
        
                # 보고걷기, 스윙, 전화받기 좌, 우 각 5분씩
        # tester1 검증 데이터
        # os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_left04.csv"),  # 5분
        # os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_right03.csv"), # 5분
        # os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left05.csv"),
        # os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right05.csv"),
        # os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left03.csv"),
        # os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right03.csv"),
        
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_lr01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_looking", "looking_lr02.csv"), # 10분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_left06.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_swing", "swing_right06.csv"), # 5분 
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_left06.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester1", "learn_calling", "calling_right06.csv"), # 5분
        
        # tester2 검증 데이터 
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "looking_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "swing_jw_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester2", "calling_jw_r01.csv"), # 5분
        
        # tester3 검증 데이터
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "looking_jh_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "swing_jh_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester3", "calling_jh_r01.csv"), # 5분
        
        # tester4 검증 데이터 
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "looking_hr_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "swing_hr_r01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_l01.csv"), # 5분
        os.path.join(BASE_DIR, "data", "tester4", "calling_hr_r01.csv"), # 5분
        
    ]
    
    
    X_list, Y_list, df_list = [], [], []
    
    for path in learn_data_paths:
        if not os.path.exists(path):
            print(f"학습 파일을 찾을 수 없습니다: {path}")
            continue

        # 파일명으로 옵션 선택
        fname = Path(path).name
        opts = config.get(fname, default_config)
        

        # 언팩하여 함수 호출
        df_all, x, y = DataProcessor.load_and_preprocess_csv(
            path, **opts, window_size=WINDOW_SIZE
        )

        df_list.append(df_all)
        X_list.append(x)
        Y_list.append(y)
        
    # 4) numpy 배열로 변환
    X = np.concatenate(X_list, axis=0)  # (batch, window_size, feature_dim)
    Y = np.concatenate(Y_list, axis=0)  # (batch, 2)
    
    # # ============================================================
    # # 5. 데이터 분포 시각화 (optional)
    # # ============================================================
    
    train_speed = Y[:, 0]
    val_speed = Y[:, 0]
    
    train_hc = np.degrees(Y[:, 1])
    val_hc = np.degrees(Y[:, 1])
    
    plt.plot(train_speed, label="Train distance")
    plt.title("Train distance (m)")
    plt.xlabel("Sample Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.show()
    
    plt.plot(val_speed, label="Validation distance", color='orange')
    plt.title("Validation distance (m)")
    plt.xlabel("Sample Index")
    plt.ylabel("Distance (m)")
    plt.grid(True)
    plt.legend()
    plt.show()
    
    plt.plot(train_hc, label="Train heading change")
    plt.title("Train heading change (degrees)")
    plt.xlabel("Sample Index")
    plt.ylabel("Heading change (degrees)")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.plot(val_hc, label="Validation heading change")
    plt.title("Validation heading change (degrees)")
    plt.xlabel("Sample Index")
    plt.ylabel("Heading change (degrees)")
    plt.grid(True)
    plt.legend()
    plt.show()
    
    def plot_pdf(data, label_name):
        plt.figure()
        plt.hist(data, bins=50, density=True, alpha=0.7)
        plt.xlabel(f"{label_name}")
        plt.ylabel("Probability Density")
        plt.title(f"PDF of {label_name}")
        plt.grid(True)
        plt.show()

    plot_pdf(train_speed, "distance (m/s)")
    plot_pdf(train_hc, "Δψ (degrees)")
    plot_pdf(val_speed, "distance (m/s)")
    plot_pdf(val_hc, "Δψ (degrees)")
    
    # # ============================================================
    # # 6. 모델 학습
    # # ============================================================
    total_samples, window_size, num_features = X.shape
    print(
        f"총 샘플 수: {total_samples}, 윈도우 크기: {window_size}, 피처 수: {num_features}"
    )
    trainer = ModelTrainer(window_size, num_features, epochs=30, batch_size=256)
    history = trainer.train_model(X, Y)
    trainer.plot_training_history(history)

    model_path = trainer.save_model()
    print("모델이 저장되었습니다:", model_path)


if __name__ == "__main__":
    main()